# 03 - Temporal Analysis

## Objective

Build and analyze sales-count and revenue time series. The analysis compares annual and monthly behavior, identifies incomplete periods, and evaluates trend and seasonality before model selection.

In [ ]:
from pathlib import Path
import pandas as pd

# Caminho of the project
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "interm"

# Load the data
csv_file = DATA_PATH / "data_2.csv"
banco = pd.read_csv(csv_file)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Parâmetros of estilo of the Gráfico based in the manual of the MBA
plt.rcParams.update({

    # font
    "font.family": "Arial",
    "font.size": 11,

    # background
    "figure.facecolor": "white",
    "axes.facecolor": "white",

    # without grid
    "axes.grid": False,

    # without title
    "axes.titlesize": 0,

    # axes
    "axes.edgecolor": "black",
    "axes.linewidth": 1.5,

    # color of the textos
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",

    # size of the titles of thexes
    "axes.labelsize": 11

})

In [ ]:
def plot_with_style(data,
                    kind='line',            # 'line' or 'bar'
                    x_label='', y_label='',
                    title='',
                    xticks=None,            # list of positions (dates or values)
                    xtlabels=None,          # list of labels for xticks
                    rotate_xticks=0,
                    figsize=(12,6),
                    marker='o',
                    linewidth=2):
    """Plota gráfico with estilo personalizado. Aceita pd.Series (index can be datetime) or pd.DataFrame."""
    fig, ax = plt.subplots(figsize=figsize)

    if isinstance(data, pd.Series):
        if kind == 'line':
            ax.plot(data.index, data.values, marker=marker, linewidth=linewidth)
        else:
            ax.bar(data.index, data.values, color='C0', edgecolor='black', linewidth=0.8)

    elif isinstance(data, pd.DataFrame):
        if kind == 'line':
            data.plot(ax=ax, marker=marker, linewidth=linewidth)
        else:
            data.plot(kind='bar', ax=ax)

    else:
        raise ValueError('data must be a pandas Series or DataFrame')

    # labels and title
    ax.set_xlabel(x_label, fontsize=11, fontname='Arial')
    ax.set_ylabel(y_label, fontsize=11, fontname='Arial')
    ax.set_title(title)

    ax.grid(False)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_linewidth(1.5)
    ax.spines['bottom'].set_linewidth(1.5)
    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_color('black')
    ax.legend(title=None, frameon=False)

    if xticks is not None:
        ax.set_xticks(xticks)
    if xtlabels is not None:
        ax.set_xticklabels(xtlabels)
    if rotate_xticks:
        plt.setp(ax.get_xticklabels(), rotation=rotate_xticks, ha='right')

    plt.tight_layout()
    plt.show()

# --- functions utilitárias for build séries temporais reutilizáveis ---
def build_monthly_counts(df, start='2017-01-01', end='2025-08-31'):
    """Returns a DataFrame indexed by month with continuous sales counts from start to end."""
    monthly = (
        df.groupby(['ano', 'mes'])
        .size()
        .reset_index(name='qtd_vendas')
    )
    monthly['date'] = pd.to_datetime(dict(year=monthly['ano'], month=monthly['mes'], day=1))
    monthly = monthly.sort_values('date').set_index('date')[['qtd_vendas']]
    idx = pd.date_range(start=start, end=end, freq='MS')
    monthly = monthly.reindex(idx, fill_value=0)
    monthly.index.name = 'date'
    return monthly

def build_annual_counts(df, start_year=2017, end_year=2025):
    """returns pd.Series index years with count of sales by year (fills years missing with 0)."""
    s = df.groupby(df['data_venda'].dt.year).size()
    years = list(range(start_year, end_year + 1))
    s = s.reindex(years, fill_value=0)
    s.index = s.index.astype(int)
    return s

def build_annual_faturamento_groups(df, groups, start_year=2017, end_year=2025):
    """Returns annual revenue totals filtered by group."""
    df_f = df[df['grupo'].isin(groups)].copy()
    s = df_f.groupby(df_f['data_venda'].dt.year)['valor_total'].sum()
    years = list(range(start_year, end_year + 1))
    s = s.reindex(years, fill_value=0)
    s.index = s.index.astype(int)
    return s

def build_monthly_faturamento_total(df, start='2017-01-01', end='2025-08-01'):
    """Returns a DataFrame indexed by month with continuous monthly revenue."""
    # Use existing calendar columns when available; otherwise extract them from data_venda
    if 'ano' in df.columns and 'mes' in df.columns:
        tmp = (
            df.groupby(['ano', 'mes'])['valor_total']
            .sum()
            .reset_index()
        )
        tmp['date'] = pd.to_datetime(dict(year=tmp['ano'], month=tmp['mes'], day=1))
    else:
        tmp = (
            df.groupby([df['data_venda'].dt.year, df['data_venda'].dt.month])['valor_total']
            .sum()
            .reset_index(name='valor_total')
        )
        tmp.columns = ['ano', 'mes', 'valor_total']
        tmp['date'] = pd.to_datetime(dict(year=tmp['ano'], month=tmp['mes'], day=1))
    monthly = tmp.sort_values('date').set_index('date')[['valor_total']]
    idx = pd.date_range(start=start, end=end, freq='MS')
    monthly = monthly.reindex(idx, fill_value=0)
    monthly.index.name = 'date'
    return monthly

def build_annual_faturamento_total(df, start_year=2017, end_year=2025):
    """Returns annual revenue totals and fills missing years with zero."""
    s = df.groupby(df['data_venda'].dt.year)['valor_total'].sum()
    years = list(range(start_year, end_year + 1))
    s = s.reindex(years, fill_value=0)
    s.index = s.index.astype(int)
    return s


In [ ]:
banco.head()

In [ ]:
banco.info()

In [ ]:
# Convert the date column to datetime
banco['data_venda'] = pd.to_datetime(
    banco['data_venda'],
    errors='coerce' # Convert invalid values to NaT
)

In [ ]:
banco['ano'] = banco['data_venda'].dt.year
banco['mes'] = banco['data_venda'].dt.month

## Annual Number of Sales

In [ ]:
# Count sales by year and month
vendas_mes_ano = (
    banco.groupby(['ano', 'mes'])
    .size()
    .reset_index(name='qtd_vendas')
)
# reset_index converts the count series into a DataFrame, 
# where the new column contains the count values.
vendas_mes_ano.head()

### Number of Sales by Month and Year

In [ ]:
pivot = vendas_mes_ano.pivot(index='mes', columns='ano', values='qtd_vendas').reindex(range(1,13))
plot_with_style(pivot, kind='line',
                x_label='month', y_label='number of sales',
                xticks=range(1,13),
                xtlabels=["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"])

In [ ]:
vendas_mes_ano.tail(10)

Monthly counts show that 2017 starts in February and that 2025 declines sharply after August. September and October 2025 were affected by a system migration and are removed to avoid artificial noise.

In [ ]:
# Removing the records of September and October of 2025
banco = banco[
    ~(
        (banco['data_venda'].dt.year == 2025)
        &
        (banco['data_venda'].dt.month.isin([9, 10]))
    )
].copy()

In [ ]:
banco.to_csv(
    "../data/interm/data_3.csv",
    index=False
)

The annual demand series is highly variable and covers only a limited number of years. Monthly sales counts are more informative for understanding timing, but aggregate revenue is a more stable forecasting target for this project.

In [ ]:
# 1) Agregar count of sales by year/mes (uso of the function utilitária)
monthly = build_monthly_counts(banco, start='2017-01-01', end='2025-08-01')

# 2) Plot usando a function of estilo
plot_with_style(
    monthly['qtd_vendas'],
    kind='line',
    x_label='year',
    y_label='number of sales',
    title='',
    xticks=pd.date_range('2017-01-01','2025-01-01',freq='YS'),
    xtlabels=[d.year for d in pd.date_range('2017-01-01','2025-01-01',freq='YS')],
    marker=''
)

In [ ]:
# aggregation: number of sales by year (each linha = 1 sale)
annual = banco.set_index('data_venda').resample('YE').size().rename('num_vendas')
annual.index = annual.index.year

# Plot
plt.figure(figsize=(10,5))
sns.lineplot(x=annual.index, y=annual.values, marker='')
plt.title('number of sales by year')
plt.xlabel('year')
plt.ylabel('number of sales')
plt.xticks(annual.index)
plt.grid(False)
plt.tight_layout()
plt.show()

The monthly sales series is valid, but forecasting total demand across all products is not sufficiently actionable because products have very different sales mechanisms. Revenue is therefore selected as the primary target.

## Annual Revenue by Group

In [ ]:
faturamento_grupo_ano = banco.groupby([banco['data_venda'].dt.year,'grupo'])['valor_total'].sum().reset_index()

faturamento_grupo_ano.columns = ['ano','grupo','revenue']

pivot = faturamento_grupo_ano.pivot(
    index='ano',
    columns='grupo',
    values='revenue'
)

fig, ax = plt.subplots(figsize=(12, 6))

pivot.plot(
    ax=ax,
    marker='o',
    linewidth=2
)

# titles of thexes
ax.set_xlabel("year", fontsize=11, fontname="Arial")
ax.set_ylabel("revenue (R$)", fontsize=11, fontname="Arial")

# Remove the title
ax.set_title("")

# Remove a grid
ax.grid(False)

# Remove bordas superior and direita
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# keeps only thexes principais
ax.spines["left"].set_linewidth(1.5)
ax.spines["bottom"].set_linewidth(1.5)

ax.spines["left"].set_color("black")
ax.spines["bottom"].set_color("black")

# legend
ax.legend(
    title=None,
    bbox_to_anchor=(0.25, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()
plt.show()

In [ ]:
def plot_faturamento_anual(df, grupos, titulo):
    df_filtrado = df[df["grupo"].isin(grupos)].copy()

    anual = (
        df_filtrado
        .groupby([df_filtrado["data_venda"].dt.year, "grupo"])["valor_total"]
        .sum()
        .reset_index()
        .pivot(index="data_venda", columns="grupo", values="valor_total")
    )

    fig, ax = plt.subplots(figsize=(12, 6))
    anual.plot(ax=ax, marker="o", linewidth=2)

    ax.set_title(titulo)
    ax.set_xlabel("year")
    ax.set_ylabel("revenue (R$)")
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(title=None, frameon=False)
    plt.tight_layout()
    plt.show()

### Annual Revenue: INTERFONE, PORTAO, and SEGURANCA

In [ ]:
# intercom, PORTAO and SEGURANCA
plot_faturamento_anual(
    banco,
    ["INTERFONE", "PORTAO", "SEGURANCA"],
    "revenue annual: intercom, PORTAO and SEGURANCA"
)

#### Annual Revenue: ANTENA, COMPONENTES ANTENAS, and CABO

In [ ]:
plot_faturamento_anual(
    banco,
    ["ANTENA", "COMPONENTES ANTENAS", "CABO"],
    "revenue annual: antenna, COMPONENTES ANTENAS and cable"
)

The group-level revenue series highlights different business profiles. Installation-related groups move together, while length-based groups have a different quantity-to-revenue relationship.

## Total Monthly Revenue

After reviewing the sales-count series, monthly revenue is selected as the main target. It is less sensitive to product-specific units and better represents the company-wide business outcome.

In [ ]:
# 1) Agregar number of sales by year (uso of the function utilitária)
vendas_por_ano = build_annual_counts(banco, start_year=2017, end_year=2025)

# 2) Plot (bar) usando a function of estilo
plot_with_style(
    vendas_por_ano,
    kind='bar',
    x_label='year',
    y_label='number of sales',
    xticks=list(range(2017,2026)),
    xtlabels=list(range(2017,2026))
)

In [ ]:
# revenue annual total by year (uso of the function utilitária)
faturamento_ano = build_annual_faturamento_total(banco, start_year=2017, end_year=2025)

# Plot revenue annual
plot_with_style(
    faturamento_ano,
    kind='bar',
    x_label='year',
    y_label='revenue (R$)',
    xticks=list(range(2017,2026)),
    xtlabels=list(range(2017,2026))
)

Annual revenue complements the annual sales count. The 2019 decline, the pandemic effect in 2020, the more stable period from 2021 onward, and the incomplete 2025 coverage are visible in both measures.

In [ ]:
faturamento_mes = banco.groupby(banco['data_venda'].dt.month)['valor_total'].sum()
plot_with_style(
    faturamento_mes,
    kind='bar',
    x_label='month',
    y_label='revenue',
    xticks=list(range(1,13)),
    xtlabels=["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]
)

In [ ]:
fat_mes_ano = (
    banco
    .assign(
        ano=banco['data_venda'].dt.year,
        mes=banco['data_venda'].dt.month
    )
    .groupby(['ano','mes'])['valor_total']
    .sum()
    .reset_index() # .reset_index() converts the result index 
    # from groupby into regular columns
)

In [ ]:
pivot_fat = fat_mes_ano.pivot(
    index='mes',
    columns='ano',
    values='valor_total'
)

plt.figure(figsize=(12,6))
sns.heatmap(
    pivot_fat,
    annot=True,
    fmt='.0f',
    cmap='YlGnBu'
)

plt.title('revenue by month and year')
plt.show()

Monthly revenue shows a recurring pattern: it rises toward April and May and declines later in the year. The series also displays a long-term growth component.

The annual and monthly results indicate both trend and within-year seasonality. These properties motivate the decomposition and forecasting models evaluated next.

## Monthly Revenue Time Series

The monthly revenue series is evaluated for trend, seasonality, and noise.

In [ ]:
faturamento_mensal = build_monthly_faturamento_total(banco, start='2017-02-01', end='2025-08-01')['valor_total']

plot_with_style(
    faturamento_mensal,
    kind='line',
    x_label='year',
    y_label='revenue (R$)',
    xticks=pd.date_range('2017-01-01','2025-01-01',freq='YS'),
    xtlabels=[d.year for d in pd.date_range('2017-01-01','2025-01-01',freq='YS')],
    marker=''
)

### Is There Seasonality After Removing the Trend?

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.dates as mdates

#### Additive Decomposition

In [ ]:
decomp_add = seasonal_decompose(
    faturamento_mensal,
    model='additive',
    period=12
)

decomp_add.plot();

In [ ]:
def plot_decomposition(series, decomposition,
                       title='Additive decomposition of monthly revenue',
                       start='2017-01-01', end='2025-01-01',
                       figsize=(12,9), grid_alpha=0.3):
    import matplotlib.dates as mdates
    fig, axes = plt.subplots(4, 1, figsize=figsize, sharex=True)

    axes[0].plot(series.index, series.values, linewidth=2)
    axes[0].set_ylabel('revenue monthly')

    axes[1].plot(decomposition.trend.index, decomposition.trend.values, linewidth=2)
    axes[1].set_ylabel('trend')

    axes[2].plot(decomposition.seasonal.index, decomposition.seasonal.values, linewidth=2)
    axes[2].set_ylabel('seasonal')

    axes[3].plot(decomposition.resid.index, decomposition.resid.values, linewidth=2)
    axes[3].set_ylabel('Residuo')

    for ax in axes:
        ax.grid(True, alpha=grid_alpha)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.5)
        ax.spines['left'].set_color('black')
        ax.spines['bottom'].set_color('black')
        ax.legend([], frameon=False)

    years = pd.date_range(start, end, freq='YS')
    axes[-1].xaxis.set_major_locator(mdates.YearLocator())
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    axes[-1].set_xticks(years)
    axes[-1].set_xticklabels([d.year for d in years])
    #plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout(rect=[0,0,1,0.96])
    plt.show()
    return fig, axes


In [ ]:
plot_decomposition(faturamento_mensal, decomp_add)

#### Multiplicative Decomposition

In [ ]:
decomp_mul = seasonal_decompose(
    faturamento_mensal,
    model='multiplicative',
    period=12
)

decomp_mul.plot();

In [ ]:
def plot_decomposition(series, decomposition,
                       title='Multiplicative decomposition of monthly revenue',
                       start='2017-01-01', end='2025-01-01',
                       figsize=(12,9), grid_alpha=0.3):
    import matplotlib.dates as mdates
    fig, axes = plt.subplots(4, 1, figsize=figsize, sharex=True)

    axes[0].plot(series.index, series.values, linewidth=2)
    axes[0].set_ylabel('revenue monthly')

    axes[1].plot(decomposition.trend.index, decomposition.trend.values, linewidth=2)
    axes[1].set_ylabel('trend')

    axes[2].plot(decomposition.seasonal.index, decomposition.seasonal.values, linewidth=2)
    axes[2].set_ylabel('seasonal')

    axes[3].plot(decomposition.resid.index, decomposition.resid.values, linewidth=2)
    axes[3].set_ylabel('Residuo')

    for ax in axes:
        ax.grid(True, alpha=grid_alpha)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_linewidth(1.5)
        ax.spines['bottom'].set_linewidth(1.5)
        ax.spines['left'].set_color('black')
        ax.spines['bottom'].set_color('black')
        ax.legend([], frameon=False)

    years = pd.date_range(start, end, freq='YS')
    axes[-1].xaxis.set_major_locator(mdates.YearLocator())
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    axes[-1].set_xticks(years)
    axes[-1].set_xticklabels([d.year for d in years])
    #plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout(rect=[0,0,1,0.96])
    plt.show()
    return fig, axes


In [ ]:
plot_decomposition(faturamento_mensal, decomp_mul)

## Conclusion

Both additive and multiplicative decompositions indicate a trend and recurring seasonal behavior. The series is not purely random, so Naive, Holt-Winters, ARIMA/SARIMA, and XGBoost models are evaluated in the next notebooks.